# Xhakinha Clips AI — notebook de operação Kaggle (v2, 18/11/2026)

## ORDEM DE EXECUÇÃO — LEIA ANTES
Execute **de cima para baixo, uma célula por vez**, esperando o fim de cada uma. As células **2–13** são diagnósticas; a célula 1 sincroniza/instala dependências. As células 5–11 são somente leitura no banco oficial; a etapa 10 só grava em cópia isolada.

| Nº | Célula | Tipo | Pode repetir? |
|---|---|---|---|
| 1 | Inicialização e sincronização do repositório | prepara ambiente (clone/pull + pip, 1x por kernel) | sim (não refaz o que já está feito) |
| 2 | Secrets (OK/AUSENTE) | só lê Secrets | sim |
| 3 | Healthcheck local | teste | sim |
| 4 | Imports e dependências | teste | sim |
| 5 | Banco SQLite local + Turso (somente leitura) | diagnóstico | sim |
| 6 | Coordenador `/api/state` | diagnóstico | sim |
| 7 | Google Drive: quota, pastas e snapshots | diagnóstico (só lê) | sim |
| 8 | Lock do Brain e executores | diagnóstico (~20 s) | sim |
| 9 | Snapshot restaurado em CÓPIA isolada | teste (baixa 1 arquivo do Drive; não altera nada) | sim |
| 10 | Fontes e scheduler (dry-run em cópia) | teste | sim |
| 11 | Idempotência (nada mudou) | teste | sim |
| 12 | Relatório consolidado com trava fechada | relatório; só libera com todos os pré-requisitos comprovados | sim |
| 13 | Status (só `/api/state`) | monitoramento | sim, quantas vezes quiser |
| 14 | Limpeza de executores | **DESATIVADA (sem ação)** | não executa alterações |
| 15 | **INICIALIZAÇÃO REAL (execução longa, ~10,5 h)** | **inicia processos** | **UMA vez; só depois de 1–12 OK** |

**A etapa 12 só libera se Turso, snapshot/restore, fonte ativa/autorizada, coordenador, lock e idempotência estiverem comprovados. Qualquer PENDENTE, AVISO ou FALHOU bloqueia.**

**NÃO execute a célula 15 para "ver o status" — use a 13.** Cada chamada de `kaggle_runner.main()` registra um executor novo no coordenador
(é isso que gerou os vários `kaggle-cpu-cc692fc6b5ef-…`). Se o kernel travar, use *Run → Restart session* e recomece pela célula 1.

## 1) Inicialização e sincronização do repositório — prepara o ambiente (clone/pull + pip; idempotente)

In [37]:
import os, sys, subprocess, base64, hashlib, json, time, socket, builtins
from pathlib import Path

REPO   = "https://github.com/gav74742-star/Xhakinha.git"
BRANCH = "main"
BASE   = Path(os.environ.get("XH_NB_BASE", "/kaggle/working/app"))
CLONE  = BASE / "xhakinha-clips-ai"

XH = globals().setdefault("XH", {"res": {}})

def rec(nome, status, msg=""):
    """Registra e imprime o resultado de um teste. status: PASSOU | FALHOU | PENDENTE | AVISO"""
    XH["res"][nome] = (status, msg)
    print(f"[{status}] {nome}" + (f" — {msg}" if msg else ""))

def precisa_celula1():
    if not XH.get("root"):
        raise SystemExit("Rode a célula 1 primeiro (e ela precisa terminar com PASSOU).")

def achar_raiz(base):
    """Raiz REAL do projeto = pasta que tem requirements-kaggle.txt e xhakinha/cloud/brain.py (não assume estrutura)."""
    for c in (base, base / "xhakinha-clips-ai", base / "xhakinha-clips-ai" / "xhakinha-clips-ai"):
        if (c / "requirements-kaggle.txt").is_file() and (c / "xhakinha" / "cloud" / "brain.py").is_file():
            return c
    return None

def git(*args, env=None, cwd=None, timeout=240):
    r = subprocess.run(["git", *args], capture_output=True, text=True, env=env, cwd=cwd, timeout=timeout)
    return r.returncode, (r.stdout + r.stderr).strip()

# --- autenticação GitHub sem token na URL, sem imprimir, sem gravar em arquivo
env = os.environ.copy()
try:
    from kaggle_secrets import UserSecretsClient
    tok = UserSecretsClient().get_secret("GITHUB_TOKEN")
    auth = base64.b64encode(f"x-access-token:{tok}".encode()).decode()
    env.update({"GIT_CONFIG_COUNT": "1", "GIT_CONFIG_KEY_0": "http.https://github.com/.extraheader",
                "GIT_CONFIG_VALUE_0": f"AUTHORIZATION: basic {auth}"})
    print("GITHUB_TOKEN: OK (não exibido)")
except Exception:
    print("GITHUB_TOKEN: AUSENTE — tentando sem autenticação (só funciona se o repositório for público)")

BASE.mkdir(parents=True, exist_ok=True)

# Preservar somente a alteração local conhecida em analysis.py.
# Qualquer outra alteração local bloqueia o pull; nunca usar reset/clean.
analysis_rel = "xhakinha-clips-ai/xhakinha/analysis.py"
analysis_path = CLONE / analysis_rel
analysis_backup = None
analysis_hash = None

if (CLONE / ".git").is_dir():
    rc, branch_atual = git("-C", str(CLONE), "rev-parse", "--abbrev-ref", "HEAD")
    if rc != 0 or branch_atual != BRANCH:
        rec("1. Repositório", "FALHOU",
            f"branch atual={branch_atual or 'indisponível'}; esperado={BRANCH}")
        raise SystemExit("Branch inesperada. Sincronização bloqueada.")

    rc, status = git("-C", str(CLONE), "status", "--porcelain", "--untracked-files=all")
    if rc != 0:
        rec("1. Repositório", "FALHOU", "não foi possível consultar git status")
        raise SystemExit("Estado Git não verificável.")

    linhas = [ln for ln in status.splitlines() if ln.strip()]
    caminhos = [ln[3:].strip() for ln in linhas]
    inesperados = [p for p in caminhos if p != analysis_rel]

    if inesperados:
        rec("1. Repositório", "FALHOU",
            "há alterações locais além de analysis.py: " + ", ".join(inesperados[:8]))
        raise SystemExit(
            "Pull bloqueado para proteger alterações locais não reconhecidas."
        )

    if analysis_rel in caminhos:
        if not analysis_path.is_file():
            raise SystemExit("analysis.py aparece modificado, mas não foi encontrado.")
        import shutil
        analysis_backup = Path("/kaggle/working") / (
            f"analysis.py.preservado-{time.time_ns()}"
        )
        shutil.copy2(analysis_path, analysis_backup)
        analysis_hash = hashlib.sha256(analysis_backup.read_bytes()).hexdigest()

        # Restaurar temporariamente apenas esse arquivo ao conteúdo do HEAD,
        # para permitir fast-forward; a cópia local fica preservada fora do clone.
        show = subprocess.run(
            ["git", "-C", str(CLONE), "show", f"HEAD:{analysis_rel}"],
            capture_output=True
        )
        if show.returncode != 0:
            shutil.copy2(analysis_backup, analysis_path)
            raise SystemExit("Não foi possível obter analysis.py do HEAD; backup preservado.")
        analysis_path.write_bytes(show.stdout)

    rc, out = git("-C", str(CLONE), "pull", "--ff-only", "origin", BRANCH, env=env)
    print("git pull:", "OK" if rc == 0 else "FALHOU — " + out[-200:])
    if rc != 0:
        if analysis_backup and analysis_backup.is_file():
            import shutil
            shutil.copy2(analysis_backup, analysis_path)
        rec("1. Repositório", "FALHOU",
            "git pull --ff-only falhou; arquivo local analysis.py foi preservado.")
        raise SystemExit("Sincronização do GitHub falhou. Não continue.")

    if analysis_backup and analysis_backup.is_file():
        import shutil
        shutil.copy2(analysis_backup, analysis_path)
        hash_restaurado = hashlib.sha256(analysis_path.read_bytes()).hexdigest()
        if hash_restaurado != analysis_hash:
            raise RuntimeError("SHA256 de analysis.py divergiu após restauração.")
        XH["analysis_backup"] = str(analysis_backup)
        XH["analysis_sha256"] = analysis_hash
        rec("1. analysis.py", "PASSOU",
            f"cópia preservada e SHA256 conferido: {analysis_hash[:16]}…")
    else:
        rec("1. analysis.py", "PASSOU",
            "nenhuma alteração local em analysis.py precisou ser preservada.")

elif CLONE.exists():
    rec("1. Repositório", "FALHOU",
        f"{CLONE} existe, mas não é um clone Git. Nada foi alterado.")
    raise SystemExit
else:
    rc, out = git("clone", "--depth", "1", "--branch", BRANCH, REPO, str(CLONE), env=env)
    if rc != 0:
        msg = out[-200:]
        if "tok" in globals() and tok:
            msg = msg.replace(tok, "***")
        rec("1. Repositório", "FALHOU", "clone falhou: " + msg)
        XH["repo_sync_ok"] = False
        raise SystemExit

ROOT = achar_raiz(CLONE)
if ROOT is None:
    rec("1. Repositório", "FALHOU",
        "não achei requirements-kaggle.txt + xhakinha/cloud/brain.py dentro do clone")
    raise SystemExit

XH["root"] = ROOT
_, br = git("-C", str(CLONE), "rev-parse", "--abbrev-ref", "HEAD")
_, sha = git("-C", str(CLONE), "rev-parse", "--short", "HEAD")
print("RAIZ DO PROJETO:", ROOT)
print("BRANCH/COMMIT  :", br, sha)
XH["repo_sync_ok"] = (br == BRANCH and bool(sha))
rec("1. Repositório", "PASSOU" if XH["repo_sync_ok"] else "FALHOU",
    f"branch={br} commit={sha}")
if not XH["repo_sync_ok"]:
    raise SystemExit("O clone não está na branch main. Não continue.")

# --- dependências: UMA vez por kernel (e só se o requirements mudou)
req = ROOT / "requirements-kaggle.txt"
h = hashlib.sha256(req.read_bytes()).hexdigest()
if XH.get("deps_hash") == h:
    print("Dependências: já instaladas neste kernel (pulando).")
else:
    r = subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", str(req)], capture_output=True, text=True)
    if r.returncode == 0:
        XH["deps_hash"] = h
        print("Dependências: instaladas.")
    else:
        print("pip FALHOU:", r.stderr[-400:])
XH["deps_ok"] = (XH.get("deps_hash") == h)
rec("1. Dependências", "PASSOU" if XH["deps_ok"] else "FALHOU", "requirements-kaggle.txt")
if not XH["deps_ok"]:
    raise SystemExit("Dependências não foram instaladas corretamente. Não continue.")

# --- sys.path: só a raiz real (remove caminhos duplicados/errados do clone)
sp = [p for p in sys.path if not str(p).startswith(str(BASE)) or str(p) == str(ROOT)]
sys.path[:] = sp
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
for m in [m for m in sys.modules if m == "xhakinha" or m.startswith("xhakinha.")]:
    f = getattr(sys.modules[m], "__file__", "") or ""
    if f and not f.startswith(str(ROOT)):
        print("AVISO: módulo xhakinha já importado de OUTRO caminho:", f, "→ use Run > Restart session e rode a célula 1 de novo.")
        break

try:
    import libsql
    XH["tem_turso"] = True
except Exception:
    XH["tem_turso"] = False
print("Driver libsql disponível:", "SIM" if XH["tem_turso"] else "NÃO")

# --- helpers de coordenador usados por várias células (somente leitura)
def coord():
    from xhakinha.cloud.client import HttpClient
    return HttpClient(os.environ["XH_COORD_URL"], os.environ["XH_COORD_TOKEN"], timeout=20)

def mostrar_estado(st, titulo="ESTADO DO COORDENADOR"):
    now = st.get("now", time.time())
    print(f"--- {titulo} ---")
    print("tarefas por status:", st.get("counts"))
    for l in st.get("locks", []):
        print(f"lock '{l['name']}': dono={l['owner']} fence={l['fence']} lease restante={round(l['lease_until'] - now)}s")
    if not st.get("locks"):
        print("locks: nenhum")
    ex = st.get("executors", [])
    print(f"executores ({len(ex)}):")
    for e in ex:
        print(f"  {e['id']:<34} {e['status']:<9} visto há {e.get('seen_ago_s')}s  restante={e.get('time_left_s')}s")
    tk = st.get("tasks", [])
    print("tarefas abertas:", len(tk), "|", {t['owner'] or '-': 0 for t in tk} if tk else "nenhuma")
    for ev in st.get("events", [])[:5]:
        print("  evento:", ev.get("level"), ev.get("component"), str(ev.get("message"))[:100])
print("Célula 1 concluída.")


GITHUB_TOKEN: OK (não exibido)
git pull: OK
RAIZ DO PROJETO: /kaggle/working/app/xhakinha-clips-ai/xhakinha-clips-ai
BRANCH/COMMIT  : main e24e7f6
[PASSOU] 1. Repositório — branch=main commit=e24e7f6
Dependências: já instaladas neste kernel (pulando).
[PASSOU] 1. Dependências — requirements-kaggle.txt
Código com suporte a Turso (dbdriver.py): NÃO — este clone é a versão antiga; o estado fica no SQLite local do kernel
Célula 1 concluída.


## 2) Secrets — somente lê (imprime só OK/AUSENTE; nunca o valor)

In [38]:
precisa_celula1()

OBRIGATORIOS = [
    "SECRET_KEY", "XH_COORD_URL", "XH_COORD_TOKEN",
    "GDRIVE_CLIENT_ID", "GDRIVE_CLIENT_SECRET", "GDRIVE_REFRESH_TOKEN"
]
OPCIONAIS = [
    "YOUTUBE_API_KEY", "TWITCH_CLIENT_ID", "TWITCH_CLIENT_SECRET",
    "TIKTOK_CLIENT_KEY", "TIKTOK_CLIENT_SECRET",
    "TURSO_DATABASE_URL", "TURSO_AUTH_TOKEN", "PUBLIC_BASE_URL"
]

try:
    from kaggle_secrets import UserSecretsClient
    cli = UserSecretsClient()
except Exception:
    cli = None

faltam = []
for nome in OBRIGATORIOS + OPCIONAIS:
    if not os.environ.get(nome) and cli:
        try:
            valor = cli.get_secret(nome)
            if valor:
                os.environ[nome] = valor
        except Exception:
            pass
    presente = bool(os.environ.get(nome))
    tipo = "obrigatório" if nome in OBRIGATORIOS else "opcional"
    print(f"{nome:<24} {'OK' if presente else 'AUSENTE':<8} ({tipo})")
    if nome in OBRIGATORIOS and not presente:
        faltam.append(nome)

# O notebook protegido sempre exige o banco compartilhado Turso.
os.environ["DATA_DIR"] = os.environ.get("DATA_DIR", "/kaggle/working/xh-data")
os.environ["XH_ENV"] = "production"
os.environ["COOKIE_SECURE"] = "1"
os.environ["XH_STATE_BACKEND"] = "turso"

from xhakinha.config import load_settings
from xhakinha.db import Store

XH["backend"] = "indeterminado"
XH["turso_ready"] = False

try:
    settings_secret = load_settings(validate=False)
    driver_ok = bool(XH.get("tem_turso"))
    credenciais_ok = bool(
        os.environ.get("TURSO_DATABASE_URL")
        and os.environ.get("TURSO_AUTH_TOKEN")
    )
    solicitado = Store.remote_requested()
    store_secret = Store(settings_secret)
    remoto = bool(store_secret.is_remote)
    turso_ok = driver_ok and credenciais_ok and solicitado and remoto
    XH["backend"] = "turso" if remoto else "sqlite"
    XH["turso_ready"] = turso_ok
    print("Driver libsql:", driver_ok)
    print("Credenciais Turso:", credenciais_ok)
    print("Backend solicitado:", solicitado)
    print("Store remoto efetivo:", remoto)
except Exception as exc:
    XH["settings_error"] = f"{type(exc).__name__}: {str(exc)[:160]}"
    print("Falha na validação Turso:", XH["settings_error"])
    turso_ok = False

rec("2. Secrets obrigatórios", "FALHOU" if faltam else "PASSOU",
    ("faltam: " + ", ".join(faltam)) if faltam else "todos presentes")
rec("2. Secrets TikTok",
    "PASSOU" if os.environ.get("TIKTOK_CLIENT_KEY") and os.environ.get("TIKTOK_CLIENT_SECRET") else "PENDENTE",
    "presença das duas credenciais; nenhuma chamada OAuth foi feita")
rec("2. Driver libsql", "PASSOU" if XH.get("tem_turso") else "FALHOU",
    "driver instalado e importável" if XH.get("tem_turso") else "libsql indisponível")
rec("2. Backend Turso efetivo", "PASSOU" if turso_ok else "FALHOU",
    f"backend={XH.get('backend')}; remoto={turso_ok}")

if faltam or not turso_ok:
    raise SystemExit("Secrets essenciais ou backend Turso não aprovados.")


  SECRET_KEY               OK       (obrigatório)
  XH_COORD_URL             OK       (obrigatório)
  XH_COORD_TOKEN           OK       (obrigatório)
  GDRIVE_CLIENT_ID         OK       (obrigatório)
  GDRIVE_CLIENT_SECRET     OK       (obrigatório)
  GDRIVE_REFRESH_TOKEN     OK       (obrigatório)
  YOUTUBE_API_KEY          OK       (opcional)
  TWITCH_CLIENT_ID         OK       (opcional)
  TWITCH_CLIENT_SECRET     OK       (opcional)
  TIKTOK_CLIENT_KEY        OK       (opcional)
  TIKTOK_CLIENT_SECRET     OK       (opcional)
  TURSO_DATABASE_URL       OK       (opcional)
  TURSO_AUTH_TOKEN         OK       (opcional)
  XH_STATE_BACKEND         OK       (opcional)
  PUBLIC_BASE_URL          AUSENTE  (opcional)
Backend de estado que o Brain usará: sqlite | DATA_DIR: /kaggle/working/xh-data
[PASSOU] 2. Secrets obrigatórios — todos presentes
[PASSOU] 2. Secrets TikTok — só presença; nenhuma chamada ao TikTok
[FALHOU] 2. Backend Turso efetivo — secrets=OK; driver=NÃO; backend=sqlite
BLO

In [41]:
import os
import sys

repo = "/kaggle/working/app/xhakinha-clips-ai/xhakinha-clips-ai"

os.chdir(repo)
sys.path.insert(0, os.path.dirname(repo))

os.environ["XH_STATE_BACKEND"] = "turso"

from xhakinha.config import load_settings
from xhakinha.db import Store

settings = load_settings()
db = Store(settings)

print("Backend remoto:", db.is_remote)
print("Backend solicitado:", os.environ["XH_STATE_BACKEND"])

assert db.is_remote, "ERRO: Store não selecionou Turso!"

print("\nVALIDAÇÃO TURSO: APROVADA")

Backend remoto: True
Backend solicitado: turso

VALIDAÇÃO TURSO: APROVADA


## 3) Healthcheck local — teste (ffmpeg, ffprobe, Whisper, rostos, GPU, disco)

In [ ]:
precisa_celula1()
from xhakinha.cloud.healthcheck import local_report
rep = local_report()
print(json.dumps(rep, indent=2))
rec("3. Healthcheck", "PASSOU" if rep.get("ok") else "FALHOU", f"ffmpeg={rep.get('ffmpeg')} ffprobe={rep.get('ffprobe')} gpu={rep.get('gpu')} disco_livre={rep.get('disk_free_gb')}GB")

## 4) Imports e dependências — teste

In [ ]:
precisa_celula1()
import importlib
mods = ["xhakinha.config", "xhakinha.db", "xhakinha.engine", "xhakinha.scheduler", "xhakinha.cloud.brain", "xhakinha.cloud.runner",
        "xhakinha.cloud.kaggle_runner", "xhakinha.cloud.snapshots", "xhakinha.cloud.drive_storage", "xhakinha.cloud.client",
        "xhakinha.sources.youtube", "xhakinha.sources.twitch", "xhakinha.sources.instagram_src", "xhakinha.publish.tiktok"]
bad = []
for m in mods:
    try:
        mod = importlib.import_module(m)
        f = str(getattr(mod, "__file__", ""))
        if not f.startswith(str(XH["root"])):
            bad.append(f"{m} veio de {f}")
    except Exception as e:
        bad.append(f"{m}: {type(e).__name__} {str(e)[:80]}")
for b in bad: print("  PROBLEMA:", b)
for pkg in ["requests", "starlette", "cryptography", "cv2", "numpy", "faster_whisper"] + (["libsql"] if XH.get("tem_turso") else []):
    try: importlib.import_module(pkg); print("  dep OK:", pkg)
    except Exception as e: bad.append(f"dep {pkg}: {type(e).__name__}"); print("  dep FALTA:", pkg)
rec("4. Imports/dependências", "FALHOU" if bad else "PASSOU",
    "; ".join(bad)[:300] if bad else f"{len(mods)} módulos")
rec("4. Driver libsql", "PASSOU" if XH.get("tem_turso") else "FALHOU",
    "libsql importável" if XH.get("tem_turso") else "libsql ausente")


## 5) Banco SQLite local e Turso — SOMENTE LEITURA (abre com mode=ro; não cria nem altera nada)

In [ ]:
precisa_celula1()
import sqlite3
from xhakinha.config import load_settings
from xhakinha.db import Store

S = load_settings(validate=False)
dbp = Path(S.db_path)
XH["db_path"] = dbp
print("DATA_DIR:", S.data_dir)
print("DB SQLite local:", dbp, "| existe:", dbp.exists())

TABELAS = ["sources", "contents", "jobs", "clips", "publications", "settings", "events"]
cnt = {}

if dbp.exists():
    XH["db_antes"] = (dbp.stat().st_size, dbp.stat().st_mtime_ns)
    c = sqlite3.connect(f"file:{dbp}?mode=ro", uri=True)
    try:
        qc = c.execute("PRAGMA quick_check").fetchone()[0]
        tabs = {r[0] for r in c.execute(
            "SELECT name FROM sqlite_master WHERE type='table'"
        )}
        for tabela in TABELAS:
            if tabela in tabs:
                cnt[tabela] = c.execute(
                    f"SELECT COUNT(*) FROM {tabela}"
                ).fetchone()[0]
    finally:
        c.close()
    print("quick_check:", qc, "| contagens:", cnt)
    if qc != "ok":
        rec("5. SQLite local", "FALHOU", f"quick_check={qc}")
    else:
        rec("5. SQLite local", "PASSOU", "banco local íntegro; leitura somente")
else:
    # Em backend remoto, não criar SQLite local é o comportamento esperado.
    rec("5. SQLite local", "PASSOU",
        "arquivo local ausente; backend oficial é Turso")

XH["db_local_counts"] = cnt

# Consulta somente leitura ao Turso usando o Store real do projeto.
try:
    if not XH.get("turso_ready"):
        raise RuntimeError("Turso não foi aprovado na etapa 2")
    store = Store(S)
    if not store.is_remote:
        raise RuntimeError("Store não está remoto")
    rows = store.q(
        "SELECT name FROM sqlite_master WHERE type='table' ORDER BY name"
    )
    tabelas_turso = {r["name"] for r in rows}
    tcnt = {}
    for tabela in TABELAS:
        if tabela in tabelas_turso:
            tcnt[tabela] = int(store.one(
                f"SELECT COUNT(*) AS n FROM {tabela}"
            )["n"])
    XH["turso_counts"] = tcnt
    XH["turso_sources_count"] = int(tcnt.get("sources", 0))
    XH["turso_tables"] = sorted(tabelas_turso)
    print("Turso remoto:", {k: tcnt.get(k, 0) for k in TABELAS})
    print("Tabelas encontradas:", len(tabelas_turso))
    rec("5. Turso (leitura)", "PASSOU",
        f"{len(tabelas_turso)} tabelas; sources={XH['turso_sources_count']}")
except Exception as exc:
    XH["turso_counts"] = None
    XH["turso_sources_count"] = None
    rec("5. Turso (leitura)", "FALHOU",
        f"{type(exc).__name__}: {str(exc)[:160]}")


## 6) Coordenador — `GET /api/state` com os headers corretos (somente leitura)

In [ ]:
precisa_celula1()
try:
    st = coord().state()
    XH["state"] = st
    mostrar_estado(st)
    ok = (isinstance(st, dict) and all(k in st for k in ("now", "counts", "executors", "locks"))
          and isinstance(st.get("executors"), list) and isinstance(st.get("locks"), list)
          and isinstance(st.get("counts"), dict))
    rec("6. Coordenador /api/state", "PASSOU" if ok else "FALHOU", f"executores={len(st.get('executors', []))} tarefas={st.get('counts')}")
    rec("6b. Turso via coordenador", "PENDENTE", "/api/state não comprova Turso; a conexão SQL de leitura é validada na célula 5")
    if not ok:
        raise RuntimeError("Resposta /api/state incompleta ou em formato inesperado")
except Exception as e:
    rec("6. Coordenador /api/state", "FALHOU", f"{type(e).__name__}: {str(e)[:150]}")

## 7) Google Drive — autenticação, quota, pastas e listagem de snapshots (SOMENTE LEITURA: sem upload, sem apagar, sem criar pasta)

In [ ]:
precisa_celula1()
from xhakinha.cloud.runner import drive_from_env
from xhakinha.cloud.snapshots import SnapshotManager
TMP = Path("/kaggle/working/xh-diag-tmp") if Path("/kaggle/working").exists() else Path("/tmp/xh-diag-tmp")
TMP.mkdir(parents=True, exist_ok=True)
try:
    drive = drive_from_env(os.environ)
    XH["drive"] = drive
    q = drive.quota()
    gb = lambda b: None if b is None else round(b / 1e9, 2)
    print("quota Drive: usado", gb(q.get("usage")), "GB | limite", gb(q.get("limit")), "GB")
    XH["drive_read_ok"] = True
    rec("7. Drive autenticação/quota", "PASSOU", f"usado={gb(q.get('usage'))}GB")
    arqs = drive.listdir("db/snapshots")
    gz = [f for f in arqs if f["path"].endswith(".gz")]
    print("arquivos em db/snapshots:", len(arqs), "| .gz:", len(gz))
    sm = SnapshotManager(drive, TMP / "list")
    ms = sm.manifests()
    XH["manifests"] = ms
    for m in ms[:5]:
        print("  manifesto:", m.get("file"), "| epoch", m.get("epoch"), "| ts", m.get("ts"), "| contagens", m.get("counts"))
    if ms:
        XH["snapshot_list_ok"] = True
        rec("7. Drive snapshots", "PASSOU", f"{len(ms)} manifesto(s); mais recente: {ms[0].get('file')}")
    elif gz:
        XH["snapshot_list_ok"] = False
        rec("7. Drive snapshots", "FALHOU", "há .gz mas nenhum manifesto — estado inconsistente; NÃO inicie vazio")
    else:
        XH["snapshot_list_ok"] = False
        rec("7. Drive snapshots", "FALHOU", "nenhum snapshot no Drive: não há recuperação comprovada para iniciar o Brain")
except Exception as e:
    XH["drive_read_ok"] = False
    XH["snapshot_list_ok"] = False
    rec("7. Drive autenticação/quota", "FALHOU", f"{type(e).__name__}: {str(e)[:150]}")
    rec("7. Drive snapshots", "FALHOU", "Drive não validado")

## 8) Lock do Brain e executores — diagnóstico (~20 s; NÃO adquire lock, NÃO registra executor)

In [ ]:
precisa_celula1()
try:
    c8 = coord()
    s1 = c8.state(); time.sleep(20); s2 = c8.state()
    if not all(isinstance(x, dict) and "now" in x and isinstance(x.get("locks"), list) for x in (s1, s2)):
        raise RuntimeError("formato de lock inesperado no /api/state")
    now = s2["now"]
    mostrar_estado(s2, "ESTADO ATUAL")
    L1 = {l.get("name"): l for l in s1.get("locks", [])}.get("brain")
    L2 = {l.get("name"): l for l in s2.get("locks", [])}.get("brain")
    ativo1 = bool(L1 and float(L1.get("lease_until", 0)) > float(s1["now"]))
    ativo2 = bool(L2 and float(L2.get("lease_until", 0)) > float(now))
    if ativo1 or ativo2:
        owner = (L2 or L1 or {}).get("owner", "desconhecido")
        veredito = f"BLOQUEADO: lock brain válido observado; dono={owner}. Não inicie outro Brain."
        cor = "FALHOU"
    elif L1 or L2:
        mesmo_lock_expirado = bool(
            L1 and L2
            and L1.get("owner") == L2.get("owner")
            and L1.get("fence") == L2.get("fence")
            and float(L2.get("lease_until", 0)) <= float(now)
            and float(L2.get("lease_until", 0)) <= float(L1.get("lease_until", 0))
        )
        if mesmo_lock_expirado:
            veredito = f"PASSOU: lock expirado e sem renovação nas duas leituras; dono antigo={L2.get('owner')} fence={L2.get('fence')}."
            cor = "PASSOU"
            XH["expired_lock"] = {"owner": L2.get("owner"), "fence": L2.get("fence")}
        else:
            veredito = "BLOQUEADO: lock expirado/instável ou mudou entre leituras; repita a checagem."
            cor = "AVISO"
    else:
        veredito = "PASSOU: lock brain ausente nas duas leituras separadas por 20 s."
        cor = "PASSOU"
        XH["expired_lock"] = None
    XH["lock_ok"] = (cor == "PASSOU")
    XH["lock_info"] = veredito
    print("VEREDITO:", veredito)
    ativos = [e for e in s2.get("executors", []) if e.get("status") != "stopped"]
    orfaos = [e.get("id") for e in ativos if (e.get("seen_ago_s") or 0) > 300]
    print("executores ativos:", len(ativos), "| sem heartbeat >300s:", orfaos or "nenhum")
    print("Lembrete: executor não equivale a Brain; não remova registros para liberar lock.")
    rec("8. Lock do Brain", cor, veredito)
except Exception as e:
    XH["lock_ok"] = False
    rec("8. Lock do Brain", "FALHOU", f"{type(e).__name__}: {str(e)[:150]}")

## 9) Snapshot restaurado em CÓPIA ISOLADA — teste (nunca toca no banco oficial)

In [ ]:
precisa_celula1()
import sqlite3
XH["restore_ok"] = False
if not XH.get("drive"):
    rec("9. Restore em cópia", "FALHOU", "Drive não autenticado/validado")
elif not XH.get("manifests"):
    rec("9. Restore em cópia", "FALHOU", "não há manifesto de snapshot para restaurar")
else:
    from xhakinha.cloud.snapshots import SnapshotManager
    copia = TMP / "restore-test" / "xhakinha-copia.db"
    copia.parent.mkdir(parents=True, exist_ok=True)
    if XH.get("db_path") and copia.resolve() == Path(XH["db_path"]).resolve():
        raise RuntimeError("a cópia de restore aponta para o banco oficial")
    try:
        sm = SnapshotManager(XH["drive"], TMP / "restore-dl")
        m = sm.restore_latest(copia)
        if not copia.is_file() or copia.stat().st_size == 0:
            raise RuntimeError("restore não produziu arquivo de banco")
        c = sqlite3.connect(f"file:{copia}?mode=ro", uri=True)
        qc = c.execute("PRAGMA quick_check").fetchone()[0]
        tabs = {r[0] for r in c.execute("SELECT name FROM sqlite_master WHERE type='table'")}
        obrig = {"sources", "contents", "jobs"}
        if qc != "ok" or not obrig.issubset(tabs):
            raise RuntimeError(f"banco restaurado inválido: quick_check={qc}; tabelas ausentes={sorted(obrig-tabs)}")
        n = {t: c.execute(f"SELECT COUNT(*) FROM {t}").fetchone()[0] for t in ("sources", "contents", "jobs", "clips", "publications") if t in tabs}
        c.close()
        XH["restored_copy"] = copia; XH["restored_counts"] = n
        XH["restore_ok"] = True
        print("snapshot:", m.get("file"), "| quick_check:", qc, "| contagens:", n)
        rec("9. Restore em cópia", "PASSOU", f"{m.get('file')} quick_check={qc} contagens={n}")
    except Exception as e:
        rec("9. Restore em cópia", "FALHOU", f"{type(e).__name__}: {str(e)[:200]}")

## 10) Fontes e scheduler — dry-run em cópia isolada

O teste usa uma cópia do snapshot restaurado e força temporariamente `XH_STATE_BACKEND=sqlite`. Executa dois `tick()` nessa cópia, sem iniciar threads e sem escrever no Turso. A ausência de fontes cadastradas é informativa: não comprova ingestão, mas não bloqueia os diagnósticos de infraestrutura.


In [ ]:
precisa_celula1()
import sqlite3, shutil, uuid
from xhakinha.config import load_settings
from xhakinha.engine import Engine
from xhakinha.db import Store

XH["sources_ready"] = False
XH["idempotency_real_ok"] = False
origem = XH.get("restored_copy")

if not XH.get("restore_ok") or not origem or not Path(origem).is_file():
    rec("10. Fontes/scheduler", "FALHOU",
        "snapshot restaurado e validado não está disponível")
    rec("10. Fontes cadastradas", "PENDENTE",
        "não foi possível consultar a cópia do snapshot")
else:
    # Ler a cópia restaurada, nunca o banco oficial.
    c = sqlite3.connect(f"file:{Path(origem).resolve()}?mode=ro", uri=True)
    try:
        qc_origem = c.execute("PRAGMA quick_check").fetchone()[0]
        fontes = c.execute(
            "SELECT platform,name,enabled,authorized,interval_s FROM sources"
        ).fetchall()
    finally:
        c.close()

    print("Fontes no snapshot:", len(fontes))
    for fonte in fontes[:20]:
        print(" ", fonte)

    # Criar diretório exclusivo; não apagar nem reutilizar diretório anterior.
    dry_dir = TMP / "dry" / f"exec-{uuid.uuid4().hex[:12]}" / "xh-data"
    dry_dir.mkdir(parents=True, exist_ok=False)
    dry_db = dry_dir / "xhakinha.db"
    shutil.copy2(origem, dry_db)

    backend_original = os.environ.get("XH_STATE_BACKEND")
    data_original = os.environ.get("DATA_DIR")
    try:
        # A seleção SQLite é temporária e restrita ao Engine de diagnóstico.
        os.environ["XH_STATE_BACKEND"] = "sqlite"
        os.environ["DATA_DIR"] = str(dry_dir)

        s_dry = load_settings(validate=False)
        s_dry.data_dir = dry_dir
        if Store.remote_requested():
            raise RuntimeError("dry-run ainda seleciona backend remoto")
        eng_dry = Engine(s_dry)
        if eng_dry.db.is_remote:
            raise RuntimeError("Engine do dry-run conectou ao Turso")
        if Path(s_dry.db_path).resolve() != dry_db.resolve():
            raise RuntimeError("Engine não está usando a cópia isolada")

        jobs_before = {
            int(r["id"]) for r in eng_dry.db.q(
                "SELECT rowid AS id FROM jobs"
            )
        }
        eng_dry.tick()
        jobs_after_1 = {
            int(r["id"]) for r in eng_dry.db.q(
                "SELECT rowid AS id FROM jobs"
            )
        }
        eng_dry.tick()
        jobs_after_2 = {
            int(r["id"]) for r in eng_dry.db.q(
                "SELECT rowid AS id FROM jobs"
            )
        }
        grouped = eng_dry.db.q(
            "SELECT type,status,COUNT(*) AS n FROM jobs GROUP BY type,status"
        )
        check = eng_dry.db.one("PRAGMA quick_check")
        quick_check = next(iter(check.values())) if check else None

        sem_mudanca_segundo_tick = jobs_after_1 == jobs_after_2
        XH["idempotency_real_ok"] = (
            sem_mudanca_segundo_tick and quick_check == "ok"
        )
        XH["dry_db_path"] = dry_db
        XH["dry_jobs"] = grouped

        print("Jobs antes:", len(jobs_before))
        print("Jobs após primeiro tick:", len(jobs_after_1))
        print("Jobs após segundo tick:", len(jobs_after_2))
        print("Jobs agrupados:", grouped)
        print("Quick check final:", quick_check)

        rec("10. Fontes/scheduler",
            "PASSOU" if quick_check == "ok" else "FALHOU",
            f"dois ticks em SQLite isolado; jobs={len(jobs_after_2)}; integridade={quick_check}")
        rec("11. Idempotência",
            "PASSOU" if XH["idempotency_real_ok"] else "FALHOU",
            "segundo tick não alterou IDs de jobs; quick_check=ok"
            if XH["idempotency_real_ok"] else
            "segundo tick alterou jobs ou quick_check falhou")
    except Exception as exc:
        rec("10. Fontes/scheduler", "FALHOU",
            f"{type(exc).__name__}: {str(exc)[:180]}")
        rec("11. Idempotência", "FALHOU",
            "não foi possível concluir o teste isolado")
    finally:
        if backend_original is None:
            os.environ.pop("XH_STATE_BACKEND", None)
        else:
            os.environ["XH_STATE_BACKEND"] = backend_original
        if data_original is None:
            os.environ.pop("DATA_DIR", None)
        else:
            os.environ["DATA_DIR"] = data_original

    qtd_fontes = len(fontes)
    ativas = [f for f in fontes if bool(f[2]) and bool(f[3])]
    XH["sources_ready"] = bool(ativas)
    if qtd_fontes:
        rec("10. Fontes cadastradas",
            "PASSOU" if ativas else "PENDENTE",
            f"{qtd_fontes} cadastradas; {len(ativas)} habilitadas e autorizadas")
    else:
        rec("10. Fontes cadastradas", "PENDENTE",
            "zero fontes no snapshot; diagnóstico pode prosseguir, mas não haverá ingestão")


## 11) Idempotência — confirma que as células anteriores não alteraram o banco oficial nem registraram executores

In [ ]:
precisa_celula1()
probs = []

dbp = XH.get("db_path")
if dbp and XH.get("db_antes") and Path(dbp).exists():
    depois = (Path(dbp).stat().st_size, Path(dbp).stat().st_mtime_ns)
    if depois != XH["db_antes"]:
        probs.append("banco SQLite local mudou durante os diagnósticos")
elif XH.get("backend") == "turso":
    print("SQLite local ausente: esperado no backend Turso.")

if not XH.get("idempotency_real_ok"):
    probs.append("idempotência do dry-run isolado não foi aprovada")

try:
    if not XH.get("state") or not isinstance(XH["state"].get("executors"), list):
        raise RuntimeError("baseline do coordenador ausente")
    antes = {e["id"] for e in XH["state"]["executors"]}
    estado_agora = coord().state()
    if not isinstance(estado_agora, dict) or not isinstance(estado_agora.get("executors"), list):
        raise RuntimeError("lista atual de executores inválida")
    agora = {e["id"] for e in estado_agora["executors"]}
    novos = agora - antes
    if novos:
        probs.append(f"executores novos apareceram: {sorted(novos)}")
except Exception as exc:
    probs.append(f"comparação de executores falhou: {type(exc).__name__}: {str(exc)[:100]}")

rec("11. Integridade/idempotência",
    "FALHOU" if probs else "PASSOU",
    "; ".join(probs) if probs else
    "banco oficial sem alteração local inesperada; dry-run idempotente; sem novos executores")


## 12) Relatório consolidado — PASSOU / FALHOU / PENDENTE

A lista obrigatória valida infraestrutura, Turso, coordenador, Drive, lock, restore e idempotência. `10. Fontes cadastradas` é informativo, não bloqueia a inicialização do serviço; sem fontes habilitadas e autorizadas, não haverá ingestão de novos conteúdos.


In [ ]:
precisa_celula1()

OBRIGATORIOS = [
    "1. Repositório",
    "1. analysis.py",
    "1. Dependências",
    "2. Secrets obrigatórios",
    "2. Backend Turso efetivo",
    "2. Driver libsql",
    "3. Healthcheck",
    "4. Imports/dependências",
    "5. SQLite local",
    "5. Turso (leitura)",
    "6. Coordenador /api/state",
    "7. Drive autenticação/quota",
    "7. Drive snapshots",
    "8. Lock do Brain",
    "9. Restore em cópia",
    "10. Fontes/scheduler",
    "11. Idempotência",
    "11. Integridade/idempotência"
]

print(f"{'TESTE':<38} {'STATUS':<9} DETALHE")
for nome, (status, msg) in sorted(XH["res"].items()):
    print(f"{nome:<38} {status:<9} {msg[:110]}")

faltantes = [n for n in OBRIGATORIOS if n not in XH["res"]]
nao_passaram = [
    n for n in OBRIGATORIOS
    if XH["res"].get(n, ("AUSENTE", ""))[0] != "PASSOU"
]
extras_falhos = [
    n for n, (status, _) in XH["res"].items()
    if status == "FALHOU"
]

XH["pronto"] = bool(
    not faltantes
    and not nao_passaram
    and not extras_falhos
    and XH.get("turso_ready") is True
    and XH.get("restore_ok") is True
    and XH.get("lock_ok") is True
    and XH.get("idempotency_real_ok") is True
)

print("\\nPRÉ-REQUISITOS OBRIGATÓRIOS:", len(OBRIGATORIOS))
print("AUSENTES:", faltantes or "nenhum")
print("NÃO PASSARAM:", nao_passaram or "nenhum")
print("OUTROS FALHARAM:", extras_falhos or "nenhum")
print("FONTES CADASTRADAS:", XH["res"].get("10. Fontes cadastradas", ("PENDENTE", "não verificado")))

print("\\nPRONTO PARA A CÉLULA 15:", "SIM" if XH["pronto"] else "NÃO — BLOQUEADO")
if not XH["pronto"]:
    print("A trava é fechada: qualquer requisito obrigatório não aprovado impede o início.")
else:
    print("Validações aprovadas. A célula 15 fará nova checagem do Turso, lock e executores.")


## 13) Status — só consulta `/api/state` (seguro repetir sempre; NÃO use a célula 15 para isso)

In [ ]:
precisa_celula1()
try:
    mostrar_estado(coord().state(), "STATUS AGORA")
except Exception as e:
    print("Falha ao consultar:", type(e).__name__, str(e)[:150])

## 14) Executores — somente observação
Esta etapa é intencionalmente sem ação. Não chama `deregister`, não remove registros e não tenta encerrar processos remotos.
Use a célula 13 para consultar o estado. Não limpe executores para tentar liberar o lock do Brain.

In [ ]:
precisa_celula1()
print("ETAPA 14 DESATIVADA.")
print("Nenhum executor será deregistrado, removido ou marcado como parado por este notebook.")
print("Para investigar executores antigos, use somente a célula 13 (status) e confira o lock brain.")

## 15) Inicialização automática protegida

Esta etapa é chamada ao final do fluxo do notebook. Se todas as validações obrigatórias estiverem aprovadas e não houver Brain/Executor concorrente, inicia `kaggle_runner.main()` uma única vez neste kernel. Não altere manualmente uma variável de confirmação. A execução é real e pode permanecer ativa pelo orçamento configurado na sessão Kaggle.


In [ ]:
precisa_celula1()

# ============================================================
# ETAPA 15 — INICIALIZAÇÃO AUTOMÁTICA PROTEGIDA
# Só inicia se todas as etapas obrigatórias passaram.
# ============================================================

if getattr(builtins, "__XH_MAIN_STARTED__", False):
    raise SystemExit(
        "main() já foi chamado neste kernel. "
        "Não repita; use a etapa 13 para consultar o status."
    )

if XH.get("pronto") is not True:
    raise SystemExit(
        "Etapa 12 não liberou a inicialização. "
        "Nenhum Brain ou Executor foi iniciado."
    )

for nome in ("turso_ready", "restore_ok", "lock_ok", "idempotency_real_ok"):
    if XH.get(nome) is not True:
        raise SystemExit(f"Pré-requisito crítico não aprovado: {nome}")

# Revalidar as credenciais e o backend real imediatamente antes do início.
if os.environ.get("XH_STATE_BACKEND", "").strip().lower() != "turso":
    raise SystemExit("XH_STATE_BACKEND não está em turso. Início bloqueado.")
if not (os.environ.get("TURSO_DATABASE_URL") and os.environ.get("TURSO_AUTH_TOKEN")):
    raise SystemExit("Credenciais Turso ausentes. Início bloqueado.")

from xhakinha.config import load_settings
from xhakinha.db import Store

s15 = load_settings(validate=False)
if not Store.remote_requested():
    raise SystemExit("Store.remote_requested() não confirmou Turso.")
st15 = Store(s15)
if not st15.is_remote:
    raise SystemExit("Store não está conectado ao Turso. Início bloqueado.")

# Estado atual do coordenador.
st15_state = coord().state()
if not isinstance(st15_state, dict):
    raise SystemExit("Estado inválido do coordenador.")
now15 = st15_state.get("now")
locks15 = st15_state.get("locks")
executores15 = st15_state.get("executors")

if not isinstance(now15, (int, float)):
    raise SystemExit("Relógio do coordenador ausente ou inválido.")
if not isinstance(locks15, list) or not isinstance(executores15, list):
    raise SystemExit("Locks ou executores em formato inválido.")

mostrar_estado(st15_state, "ÚLTIMA CHECAGEM ANTES DA INICIALIZAÇÃO")

# Lock brain: aceitar somente ausência ou o mesmo lock expirado diagnosticado.
brain_locks15 = [lk for lk in locks15 if lk.get("name") == "brain"]
if len(brain_locks15) > 1:
    raise SystemExit("Há registros duplicados do lock brain. Início bloqueado.")

expired15 = XH.get("expired_lock")
if brain_locks15:
    lk15 = brain_locks15[0]
    lease15 = float(lk15.get("lease_until") or 0)
    if lease15 > float(now15):
        raise SystemExit("Já existe Brain com lease ativo. Início bloqueado.")
    if not isinstance(expired15, dict):
        raise SystemExit("Lock expirado não corresponde ao diagnóstico anterior.")
    if (
        lk15.get("owner") != expired15.get("owner")
        or lk15.get("fence") != expired15.get("fence")
    ):
        raise SystemExit(
            "Lock brain mudou desde a etapa 8. "
            "Repita os diagnósticos antes de iniciar."
        )
else:
    if expired15 is not None:
        raise SystemExit(
            "Lock brain desapareceu desde a etapa 8. "
            "Repita os diagnósticos antes de iniciar."
        )

# Estados terminais conhecidos. Qualquer estado desconhecido bloqueia por segurança.
terminais15 = {"lost", "stopped", "done", "failed", "offline"}
indeterminados15 = [
    e for e in executores15
    if str(e.get("status", "")).strip().lower() not in terminais15
]
if indeterminados15:
    raise SystemExit(
        "Há executores ativos ou com estado não reconhecido. "
        f"Início bloqueado: {indeterminados15}"
    )

print("[PASSOU] Turso remoto confirmado.")
print("[PASSOU] Lock brain livre e consistente.")
print("[PASSOU] Nenhum executor ativo ou estado desconhecido.")
print("[INFO] Fontes cadastradas:",
      XH["res"].get("10. Fontes cadastradas", ("PENDENTE", "não verificado")))

# Marcar antes da chamada: se houver falha parcial, não tentar iniciar de novo
# automaticamente neste mesmo kernel.
builtins.__XH_MAIN_STARTED__ = True

from xhakinha.cloud import kaggle_runner

print("\\n[INICIANDO] Xhakinha Clips AI...")
print("[INICIANDO] kaggle_runner.main() — Brain/Executor/Engine conforme configuração.")
kaggle_runner.main()
